# 🔬 crdt-merge v0.4.0 — A100 Feature Tests

Stress-testing **crdt-merge v0.4.0** features on an NVIDIA A100 GPU runtime:
provenance tracking, CRDT verification at scale, and streaming merge optimization.

Copyright 2026 Ryan Gillespie — Apache-2.0

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q crdt-merge==0.4.0 psutil

import crdt_merge, sys, platform, psutil, torch

print(f'crdt-merge  {crdt_merge.__version__}')
assert crdt_merge.__version__.startswith('0.4'), f'Expected 0.4.x, got {crdt_merge.__version__}'
print(f'Python      {sys.version}')
print(f'Platform    {platform.platform()}')
print(f'CPU cores   {psutil.cpu_count(logical=True)}')
print(f'RAM         {psutil.virtual_memory().total / 1e9:.1f} GB')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU         {props.name}')
    print(f'VRAM        {props.total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No CUDA device detected — running CPU-only')

In [ ]:
# ── Utilities ──────────────────────────────────────────────────────────
import time, json, gc, tracemalloc
import pandas as pd, numpy as np, random

RESULTS = []

def record(suite, test, rows=None, trials=None, duration_ms=None,
           mem_mb=None, extra=None):
    entry = {'suite': suite, 'test': test}
    if rows is not None: entry['rows'] = rows
    if trials is not None: entry['trials'] = trials
    if duration_ms is not None: entry['duration_ms'] = round(duration_ms, 2)
    if mem_mb is not None: entry['mem_mb'] = round(mem_mb, 2)
    if extra is not None: entry.update(extra)
    RESULTS.append(entry)
    tag = f'{rows:>10,} rows' if rows else (f'{trials:>6,} trials' if trials else '')
    ms  = f'{duration_ms:>10,.1f} ms' if duration_ms else ''
    mb  = f'{mem_mb:>8,.1f} MB' if mem_mb else ''
    print(f'  ✔ {test:<40s} {tag}  {ms}  {mb}')

class MemTracker:
    def __enter__(self):
        gc.collect()
        tracemalloc.start()
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        self.duration_ms = (time.perf_counter() - self.t0) * 1000
        _, self.peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        self.peak_mb = self.peak / 1e6

print('Utilities loaded ✓')

## Suite 1: PROVENANCE — `merge_with_provenance` at Scale

Test `merge_with_provenance` at progressive row counts to measure throughput and memory.

In [ ]:
# ── Suite 1: Provenance at Scale ──────────────────────────────────────
from crdt_merge import merge_with_provenance, export_provenance

SCALES_PROV = [1_000, 10_000, 100_000, 500_000, 1_000_000]

for n in SCALES_PROV:
    overlap = n // 2
    ids_a = list(range(n))
    ids_b = list(range(overlap, overlap + n))
    df_a = pd.DataFrame({
        'id': ids_a,
        'value': np.random.randint(0, 1000, n),
        'ts': pd.date_range('2025-01-01', periods=n, freq='s')
    })
    df_b = pd.DataFrame({
        'id': ids_b,
        'value': np.random.randint(0, 1000, n),
        'ts': pd.date_range('2025-06-01', periods=n, freq='s')
    })

    with MemTracker() as mt:
        result_df, prov = merge_with_provenance(df_a, df_b, key='id', timestamp_col='ts')

    assert prov.merged_rows >= 0
    assert prov.total_rows > 0
    assert prov.duration_ms >= 0

    prov_json = export_provenance(prov, format='json')
    assert isinstance(json.loads(prov_json), dict)

    record('PROVENANCE', f'merge_with_provenance({n:,})',
           rows=n*2, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={
               'merged_rows': prov.merged_rows,
               'total_conflicts': prov.total_conflicts,
               'unique_a': prov.unique_a_rows,
               'unique_b': prov.unique_b_rows,
           })

    del df_a, df_b, result_df, prov
    gc.collect()

print(f'\nSuite 1 complete — {len(SCALES_PROV)} scale tests ✓')

## Suite 2: VERIFY_AT_SCALE — Verification at High Trial Counts

Verify CRDT properties (commutativity, associativity, idempotency, convergence)
at escalating trial counts.

In [ ]:
# ── Suite 2: Verification at Scale ────────────────────────────────────
from crdt_merge.verify import (
    verify_crdt, verify_convergence,
    verify_commutative, verify_associative, verify_idempotent
)

def gen_fn():
    n = random.randint(5, 20)
    return pd.DataFrame({
        'id': range(n),
        'value': [random.randint(0, 100) for _ in range(n)]
    })

def merge_fn(a, b):
    return crdt_merge.merge(a, b, key='id')

# Full verify_crdt at progressive trial counts
TRIAL_COUNTS = [100, 500, 1000, 5000]
for t in TRIAL_COUNTS:
    with MemTracker() as mt:
        result = verify_crdt(merge_fn, gen_fn, trials=t)

    assert result.commutativity.passed, 'Commutativity failed'
    assert result.associativity.passed, 'Associativity failed'
    assert result.idempotency.passed, 'Idempotency failed'
    assert result.convergence.passed, 'Convergence failed'

    record('VERIFY', f'verify_crdt(trials={t})',
           trials=result.total_trials, duration_ms=mt.duration_ms,
           mem_mb=mt.peak_mb,
           extra={'total_duration_ms': result.total_duration_ms})
    gc.collect()

# Individual property checks at 2000 trials
for name, fn in [('commutative', verify_commutative),
                  ('associative', verify_associative),
                  ('idempotent', verify_idempotent),
                  ('convergence', verify_convergence)]:
    kw = {'trials': 2000}
    if name == 'convergence':
        kw['num_replicas'] = 5
    with MemTracker() as mt:
        vr = fn(merge_fn, gen_fn, **kw)
    assert vr.passed, f'{name} failed'
    record('VERIFY', f'verify_{name}(2000)',
           trials=vr.trials, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb)
    gc.collect()

print(f'\nSuite 2 complete ✓')

## Suite 3: STREAMING_OPTIMIZATION — `merge_sorted_stream` + `StreamStats`

Benchmark streaming merge at progressive row counts up to 10M.

In [ ]:
# ── Suite 3: Streaming Merge ──────────────────────────────────────────
from crdt_merge import merge_sorted_stream, StreamStats

SCALES_STREAM = [10_000, 100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]

def sorted_iter(n, offset=0):
    """Generate sorted dicts by id."""
    for i in range(n):
        yield {'id': offset + i, 'value': random.randint(0, 10000)}

for n in SCALES_STREAM:
    overlap = n // 2
    stats = StreamStats()

    with MemTracker() as mt:
        total_rows = 0
        for batch in merge_sorted_stream(
            sorted_iter(n, 0),
            sorted_iter(n, overlap),
            key='id',
            batch_size=5000,
            stats=stats
        ):
            total_rows += len(batch)

    record('STREAMING', f'merge_sorted_stream({n:,})',
           rows=n*2, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={
               'output_rows': total_rows,
               'batches': stats.batches_processed,
               'rows_per_sec': round(stats.rows_per_sec, 1) if stats.rows_per_sec else 0,
               'peak_batch': stats.peak_batch_size,
           })

    gc.collect()

print(f'\nSuite 3 complete — {len(SCALES_STREAM)} scale tests ✓')

## Suite 4: SYSTEM_SANITY — Quick End-to-End Validation

Rapid smoke tests of core CRDTs and merge functions.

In [ ]:
# ── Suite 4: Sanity Checks ────────────────────────────────────────────
from crdt_merge import GCounter, PNCounter, LWWRegister, merge

checks = 0

# 1. Basic merge
df1 = pd.DataFrame({'id': [1,2,3], 'v': [10,20,30]})
df2 = pd.DataFrame({'id': [2,3,4], 'v': [25,35,40]})
merged = merge(df1, df2, key='id')
assert len(merged) == 4; checks += 1
print(f'  ✔ merge() basic — {len(merged)} rows')

# 2. merge_dicts
from crdt_merge import merge_dicts
d1 = [{'id': 1, 'x': 10}]
d2 = [{'id': 1, 'x': 20}]
md = merge_dicts(d1, d2, key='id')
assert len(md) == 1; checks += 1
print(f'  ✔ merge_dicts() — {len(md)} rows')

# 3. GCounter
gc1 = GCounter()
gc1.increment('a', 5)
gc1.increment('b', 3)
assert gc1.value == 8; checks += 1
print(f'  ✔ GCounter value={gc1.value}')

# 4. PNCounter
pn = PNCounter()
pn.increment('a', 10)
pn.decrement('a', 3)
assert pn.value == 7; checks += 1
print(f'  ✔ PNCounter value={pn.value}')

# 5. LWWRegister
reg = LWWRegister(value='hello')
reg.set('world', 'node1')
assert reg.value == 'world'; checks += 1
print(f'  ✔ LWWRegister value={reg.value}')

# 6. GCounter initial
gc2 = GCounter(initial=100)
assert gc2.value == 100; checks += 1
print(f'  ✔ GCounter(initial=100) value={gc2.value}')

# 7. PNCounter decrement below zero
pn2 = PNCounter()
pn2.decrement('x', 5)
assert pn2.value == -5; checks += 1
print(f'  ✔ PNCounter negative value={pn2.value}')

# 8. LWWRegister with timestamp
reg2 = LWWRegister(value='v1', timestamp=100)
assert reg2.value == 'v1'; checks += 1
print(f'  ✔ LWWRegister(timestamp) value={reg2.value}')

# 9. merge with timestamp_col
df_a = pd.DataFrame({'id': [1], 'v': [10], 'ts': ['2025-01-01']})
df_b = pd.DataFrame({'id': [1], 'v': [20], 'ts': ['2025-06-01']})
m = merge(df_a, df_b, key='id')
assert len(m) == 1; checks += 1
print(f'  ✔ merge() with timestamps — {len(m)} row')

# 10. Large GCounter
gc3 = GCounter()
for i in range(100):
    gc3.increment(f'node_{i}', 1)
assert gc3.value == 100; checks += 1
print(f'  ✔ GCounter 100 nodes value={gc3.value}')

record('SANITY', f'{checks} checks passed', extra={'checks': checks})
print(f'\nSuite 4 complete — {checks}/10 checks ✓')

## Results Summary

Aggregate all benchmark results.

In [ ]:
# ── Results Summary ────────────────────────────────────────────────────
print(f'Total results: {len(RESULTS)}')
print(f'{"Suite":<15s} {"Test":<45s} {"Duration (ms)":>14s} {"Mem (MB)":>10s}')
print('─' * 90)
for r in RESULTS:
    suite = r.get('suite', '')
    test  = r.get('test', '')
    ms    = f"{r['duration_ms']:,.1f}" if 'duration_ms' in r else '—'
    mb    = f"{r['mem_mb']:,.1f}" if 'mem_mb' in r else '—'
    print(f'{suite:<15s} {test:<45s} {ms:>14s} {mb:>10s}')

with open('v040_a100_results.json', 'w') as f:
    json.dump({'version': '0.4.0', 'results': RESULTS}, f, indent=2, default=str)
print('\n💾 Saved v040_a100_results.json')
print('\n🏁 All v0.4.0 A100 tests complete!')